# 01 — Reliable API Ingestion and Immutable Raw Data

## Business scenario

You joined a commerce company whose order service exposes a paginated REST API.
Analytics needs every order, but the source occasionally rate-limits clients,
returns temporary server errors, changes its JSON schema, and repeats records
across page boundaries.
- paginated REST API: 대량의 데이터를 한 번에 반환하지 않고 여러 번의 HTTP 요청을 통해 작은 묶음으로 나누어 제공하는 API

Your job is to build the **Extract** boundary of a production-like pipeline.
You will preserve raw API responses before transforming them so that an incident
can be replayed and audited.

### Learning objectives

By the end of this notebook, you can:

1. Call a real HTTP endpoint and follow cursor pagination.
   - cursor: 현재 API 호출에서 다음 페이지가 어디인지 표시
2. Distinguish retryable failures (`429`, `500`) from permanent client errors.
3. Use bounded exponential backoff and structured JSON logging.
4. Store immutable Bronze pages, checksums, source metadata, and a run manifest.
5. Maintain a high-watermark checkpoint with an atomic write.
    - High-watermark checkpoint: 성공적으로 처리한 마지막 데이터 위치를 기억하는 영구 책갈피
    - Atomic write: 책갈피 파일이 손상되거나 반만 기록되지 않도록 임시 파일을 완성한 후 한 번에 교체하는 저장 방식
6. Explain why extraction and transformation should be separate concerns.
7. Prove that a second run is incremental rather than duplicating data.

> This notebook starts a deterministic API on `127.0.0.1`. It is synthetic,
> but requests travel through the HTTP stack. This avoids API keys, cost, and
> classroom failures caused by a changing third-party service.


## Target architecture

```mermaid
flowchart LR
    A[Local commerce REST API] -->|HTTP GET + cursor| B[Extractor]
    B --> C[Bronze raw page JSON]
    B --> D[Run manifest]
    B --> E[Watermark checkpoint]
    F[Application JSONL logs] --> G[Bronze log files]
    C --> H[Notebook 02: Spark Silver layer]
    G --> H
```

**Bronze rule:** save the source representation with ingestion metadata, but do
not silently repair business fields at this boundary. A raw copy lets us replay
improved transformation logic later.

JSON is **semi-structured**, not truly unstructured: keys provide structure, but
records can be nested or inconsistent. Free-text log messages and stack traces
are unstructured fields embedded inside semi-structured events.


```text
1. build_source_orders()
   합성 주문 240개 생성
              │
              ▼
2. SOURCE_ORDERS
   합성 주문을 Python 메모리에 보관
              │
              ▼
3. ThreadingHTTPServer
   로컬 HTTP 서버 생성 및 실행
              │
              ▼
4. CommerceAPIHandler
   들어오는 HTTP 요청을 받아서 처리
              │
              ▼
5. GET /v1/orders
   주문 한 페이지를 JSON 응답으로 반환
              │
              ▼
6. get_json_with_retry()
   API를 호출하고 일시적인 오류가 발생하면 재시도
              │
              ▼
7. Bronze Layer
   API에서 받은 원본 응답을 변환하지 않고 저장
```

In [1]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import gzip
import json
import random
import shutil
import sys
import threading
import time
import uuid

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.capstone_utils import atomic_write_json, configure_json_logger, file_sha256

LAB_ROOT = PROJECT_ROOT / "lab_data"
BRONZE_API = LAB_ROOT / "bronze" / "api" / "orders"
BRONZE_LOGS = LAB_ROOT / "bronze" / "logs"
STATE_DIR = LAB_ROOT / "state"
CHECKPOINT_PATH = STATE_DIR / "orders_checkpoint.json"
LOGGER = configure_json_logger("api_ingestion")

# This reset is deliberately explicit and narrowly scoped to this project's lab_data.
START_FRESH = True
if START_FRESH and LAB_ROOT.exists():
    assert LAB_ROOT.name == "lab_data" and PROJECT_ROOT in LAB_ROOT.parents
    shutil.rmtree(LAB_ROOT)

for directory in (BRONZE_API, BRONZE_LOGS, STATE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Lab data root: {LAB_ROOT}")


Project root: C:\Users\sungj\sam_data_engineering_tutoring_capstone
Lab data root: C:\Users\sungj\sam_data_engineering_tutoring_capstone\lab_data


## Source API and Data Engineering Client

The data engineering pipeline acts as an HTTP client. It sends a request to an upstream API, receives a JSON response, and stores the unmodified response in the Bronze layer.

```text
Data Engineering Pipeline
get_json_with_retry() (데이터를 요청하고 가져오는 쪽: API Client/Ingestion pipeline)
        │
        │ HTTP GET request
        ▼
Upstream Commerce API
CommerceAPIHandler (데이터를 제공하는 쪽: Source/Upstream API server)
        │
        │ JSON response
        ▼
Data Engineering Pipeline
Save raw response to Bronze
```

In this lab, `CommerceAPIHandler` is a local API server that simulates an upstream commerce system.

In a real company, the upstream API would normally be operated by:

- an internal backend team that owns the order service;
- another department that owns a source application;
- or an external provider such as Shopify, Stripe, or Salesforce.

The data engineer typically does not generate the source orders. The data engineer builds the client-side ingestion pipeline that calls the source API, handles pagination and temporary failures, and stores the returned data reliably.

```text
This lab                              Real company

build_source_orders()                 Production application database
        │                                         │
CommerceAPIHandler                    Order service / external API
        │                                         │
        └──── JSON over HTTP ──────────────────────┘
                          │
                          ▼
                Data Engineering Pipeline
                  get_json_with_retry()
                          │
                          ▼
                     Bronze Layer
```

## Build a deterministic source system

The source contains deliberate complications:

- API schema version 1 uses `customer_id`.
- Version 2 uses a nested `customer.id`.
- Some monetary values arrive as strings.
- One page boundary repeats an order.
- The first request for selected cursors returns `429` or `500`.
  - 200 → 요청 성공
  - 404 → 요청한 주소를 찾을 수 없음
  - 429 → 요청을 너무 많이 보냄
  - 500 → 서버 내부 오류
  - 503 → 서비스 일시 중단

In [2]:
# 주문 240개(합성 데이터)를 만들어내는 함수 
def build_source_orders(count: int = 240, seed: int = 17) -> list[dict]:
    rng = random.Random(seed)
    start = datetime(2026, 1, 1, tzinfo=timezone.utc)
    records = []
    for index in range(count):
        order_id = f"ord_{index + 1:06d}"
        customer_id = f"cust_{rng.randint(1, 60):04d}" #고객번호 무작위 생성
        event_time = start + timedelta(minutes=index * 17) #17분 간격 주문 발생
        items = [
            {
                "sku": f"SKU-{rng.randint(1, 40):03d}",
                "quantity": rng.randint(1, 3),
                "unit_price": round(rng.uniform(4, 120), 2), #주문 상품 생성
            }
            for _ in range(rng.randint(1, 4))
        ]
        total = round(sum(i["quantity"] * i["unit_price"] for i in items), 2) #주문 총액 계산
        schema_version = 2 if index % 7 == 0 else 1 # schema version 변환: 실제 API가 업데이트되면서 JSON 구조가 바뀌는 schema evolution을 모방
        record = {
            "order_id": order_id,
            "event_type": "order_created",
            "event_timestamp": event_time.isoformat(),
            "updated_at": (event_time + timedelta(minutes=5)).isoformat(),
            "status": rng.choice(["pending", "paid", "shipped"]),
            "currency": "USD",
            "total_amount": str(total) if index % 19 == 0 else total, #일부 금액은 문자열
            "items": items,
            "schema_version": schema_version,
        }
        if schema_version == 1:
            record["customer_id"] = customer_id
        else:
            record["customer"] = {"id": customer_id, "tier": rng.choice(["standard", "plus"])}
        if index == 31:
            record["currency"] = None #통화값 누락
        if index == 67:
            record["total_amount"] = "not-a-number"
        records.append(record)

    # Simulate at-least-once source delivery.
    records.insert(81, json.loads(json.dumps(records[79]))) #중복 데이터 생성
    return records


SOURCE_ORDERS = build_source_orders()
len(SOURCE_ORDERS), SOURCE_ORDERS[0]


(241,
 {'order_id': 'ord_000001',
  'event_type': 'order_created',
  'event_timestamp': '2026-01-01T00:00:00+00:00',
  'updated_at': '2026-01-01T00:05:00+00:00',
  'status': 'paid',
  'currency': 'USD',
  'total_amount': '440.06',
  'items': [{'sku': 'SKU-020', 'quantity': 2, 'unit_price': 37.6},
   {'sku': 'SKU-035', 'quantity': 3, 'unit_price': 36.28},
   {'sku': 'SKU-002', 'quantity': 1, 'unit_price': 48.56},
   {'sku': 'SKU-027', 'quantity': 2, 'unit_price': 103.73}],
  'schema_version': 2,
  'customer': {'id': 'cust_0034', 'tier': 'plus'}})

In [3]:
# 합성 주문 데이터를 제공하는 로컬 가짜 REST API를 구현한다.
class CommerceAPIHandler(BaseHTTPRequestHandler):

    # cursor별 요청 횟수를 저장한다.
    # 특정 요청의 첫 번째 시도에서만 429 또는 500 오류를 발생시키기 위해 사용한다.
    attempts: dict[str, int] = {}

    def log_message(self, format, *args):
        # HTTP 서버의 기본 로그를 숨겨 노트북 출력을 간결하게 유지한다.
        return

    def _send_json(
        self,
        status: int,
        payload: dict,
        headers: dict | None = None,
    ):
        # Python dictionary를 HTTP JSON 응답으로 변환하여 전송한다.
        body = json.dumps(payload).encode("utf-8")

        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))

        # Retry-After 같은 추가 HTTP header가 있으면 함께 전송한다.
        for key, value in (headers or {}).items():
            self.send_header(key, value)

        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        # GET 요청이 들어오면 Python HTTP 서버가 자동으로 이 메서드를 호출한다.
        parsed = urlparse(self.path)

        # 이 실습에서는 /v1/orders endpoint만 제공한다.
        if parsed.path != "/v1/orders":
            self._send_json(404, {"error": "not_found"})
            return

        # URL의 query parameter를 읽는다.
        # 예: /v1/orders?cursor=40&limit=40
        query = parse_qs(parsed.query)

        # cursor가 없으면 첫 번째 데이터 위치인 0부터 시작한다.
        cursor = query.get("cursor", ["0"])[0]

        # 한 페이지에 기본 40개, 최대 100개의 주문을 반환한다.
        limit = min(
            int(query.get("limit", ["40"])[0]),
            100,
        )

        # 이전 실행 이후 변경된 주문만 가져오기 위한 high-watermark 값이다.
        updated_after = query.get("updated_after", [None])[0]

        # 동일한 요청이 몇 번째 시도인지 기록한다.
        request_key = f"{cursor}|{updated_after}"
        self.attempts[request_key] = (
            self.attempts.get(request_key, 0) + 1
        )

        # cursor=40의 첫 요청은 일부러 429를 반환하여 retry를 테스트한다.
        if cursor == "40" and self.attempts[request_key] == 1:
            self._send_json(
                429,
                {"error": "rate_limited"},
                {"Retry-After": "0.05"},
            )
            return

        # cursor=120의 첫 요청은 일부러 500을 반환하여 retry를 테스트한다.
        if cursor == "120" and self.attempts[request_key] == 1:
            self._send_json(
                500,
                {"error": "temporary_upstream_failure"},
            )
            return

        records = SOURCE_ORDERS

        # high-watermark 이후에 변경된 주문만 선택한다.
        if updated_after:
            records = [
                record
                for record in records
                if record["updated_at"] > updated_after
            ]

        # cursor부터 limit 개수만큼 잘라 현재 페이지를 만든다.
        offset = int(cursor)
        page = records[offset : offset + limit]

        # 데이터가 더 있으면 다음 페이지의 cursor를 반환한다.
        next_cursor = (
            str(offset + limit)
            if offset + limit < len(records)
            else None
        )

        # 주문 데이터와 다음 cursor를 정상 응답(200)으로 전송한다.
        self._send_json(
            200,
            {
                "data": page,
                "next_cursor": next_cursor,
                "request_id": str(uuid.uuid4()),
                "schema": "orders-envelope-v1",
            },
        )


# 127.0.0.1에서만 접속할 수 있는 로컬 HTTP 서버를 만든다.
# 포트 0은 Windows가 사용 가능한 포트를 자동으로 선택하라는 뜻이다.
api_server = ThreadingHTTPServer(
    ("127.0.0.1", 0),
    CommerceAPIHandler,
)

# 서버가 노트북 실행을 막지 않도록 백그라운드 thread에서 실행한다.
api_thread = threading.Thread(
    target=api_server.serve_forever,
    daemon=True,
)
api_thread.start()

# 자동으로 선택된 포트를 사용해 주문 API 주소를 만든다.
API_URL = (
    f"http://127.0.0.1:"
    f"{api_server.server_port}/v1/orders"
)

print(API_URL)

http://127.0.0.1:49521/v1/orders


## Design retry behavior deliberately

A retry is appropriate when a failure is likely temporary. Common examples are
`429 Too Many Requests`, `500`, `502`, `503`, `504`, connection resets, and
timeouts. Retrying most `4xx` errors is wasteful because a bad credential or
invalid request will not repair itself.

Production guardrails include:

- a maximum number of attempts;
- bounded exponential backoff;
- server-provided `Retry-After` when available;
- connect/read timeouts;
- structured logs containing attempt and request metadata;
- a final exception rather than silently returning incomplete data.


In [4]:
# 일시적인 문제일 가능성이 있어 재시도할 HTTP 상태 코드
RETRYABLE_STATUS = {429, 500, 502, 503, 504}


# 데이터 엔지니어의 API 클라이언트:
# API를 호출하고 일시적인 HTTP/네트워크 오류가 발생하면 재시도한다.
def get_json_with_retry(
    session: requests.Session,
    url: str,
    *,
    params: dict,
    max_attempts: int = 4,
    base_delay_seconds: float = 0.05,
) -> dict:

    # 최대 max_attempts 횟수만큼 API 호출을 시도한다.
    for attempt in range(1, max_attempts + 1):
        try:
            # 연결은 최대 2초, 응답은 최대 5초 동안 기다린다.
            response = session.get(
                url,
                params=params,
                timeout=(2, 5),
            )

            # 재시도 대상이 아닌 응답이면 즉시 처리한다.
            if response.status_code not in RETRYABLE_STATUS:

                # 400, 401, 404 등의 오류이면 예외를 발생시킨다.
                response.raise_for_status()

                # 200 응답이면 JSON을 Python dictionary로 변환해 반환한다.
                return response.json()

            # 마지막 시도에서도 429 또는 500 계열 오류이면 최종 실패 처리한다.
            if attempt == max_attempts:
                response.raise_for_status()

            # 서버가 Retry-After를 보내면 그 시간을 사용한다.
            # 없으면 재시도할 때마다 대기 시간을 두 배로 늘린다.
            delay = float(
                response.headers.get(
                    "Retry-After",
                    base_delay_seconds * 2 ** (attempt - 1),
                )
            )

            # 실습이 너무 오래 걸리지 않도록 최대 대기 시간을 1초로 제한한다.
            delay = min(delay, 1.0)

            # 어떤 요청을 몇 번째로 재시도하는지 JSON 로그로 기록한다.
            LOGGER.warning(
                "retryable HTTP response",
                extra={
                    "step": "extract",
                    "attempt": attempt,
                    "source": response.url,
                },
            )

            time.sleep(delay)

        # 응답 시간 초과 또는 네트워크 연결 실패도 재시도한다.
        except (requests.Timeout, requests.ConnectionError):

            # 마지막 시도에서도 실패하면 원래 예외를 밖으로 전달한다.
            if attempt == max_attempts:
                raise

            # 0.05초 → 0.1초 → 0.2초처럼 대기 시간을 늘린다.
            delay = min(
                base_delay_seconds * 2 ** (attempt - 1),
                1.0,
            )

            LOGGER.warning(
                "retryable network exception",
                extra={
                    "step": "extract",
                    "attempt": attempt,
                    "source": url,
                },
            )

            time.sleep(delay)

    # 정상적인 코드 흐름에서는 여기에 도달하지 않는다.
    raise RuntimeError("unreachable: retry loop exhausted")


# Session을 사용하면 여러 API 요청에서 HTTP 연결을 재사용할 수 있다.
with requests.Session() as session:

    # 주문 API에 2개의 주문을 요청하여 retry 함수를 간단히 테스트한다.
    sample = get_json_with_retry(
        session,
        API_URL,
        params={"limit": 2},
    )

sample

{'data': [{'order_id': 'ord_000001',
   'event_type': 'order_created',
   'event_timestamp': '2026-01-01T00:00:00+00:00',
   'updated_at': '2026-01-01T00:05:00+00:00',
   'status': 'paid',
   'currency': 'USD',
   'total_amount': '440.06',
   'items': [{'sku': 'SKU-020', 'quantity': 2, 'unit_price': 37.6},
    {'sku': 'SKU-035', 'quantity': 3, 'unit_price': 36.28},
    {'sku': 'SKU-002', 'quantity': 1, 'unit_price': 48.56},
    {'sku': 'SKU-027', 'quantity': 2, 'unit_price': 103.73}],
   'schema_version': 2,
   'customer': {'id': 'cust_0034', 'tier': 'plus'}},
  {'order_id': 'ord_000002',
   'event_type': 'order_created',
   'event_timestamp': '2026-01-01T00:17:00+00:00',
   'updated_at': '2026-01-01T00:22:00+00:00',
   'status': 'shipped',
   'currency': 'USD',
   'total_amount': 21.51,
   'items': [{'sku': 'SKU-009', 'quantity': 1, 'unit_price': 21.51}],
   'schema_version': 1,
   'customer_id': 'cust_0009'}],
 'next_cursor': '2',
 'request_id': '9bed291a-1deb-4e32-976e-e9231b163d15'

In [5]:
print("=== 네트워크 연결 실패 테스트 ===")

try:
    with requests.Session() as session:
        get_json_with_retry(
            session,
            "http://127.0.0.1:1/v1/orders",
            params={},
            max_attempts=3,
        )

except requests.RequestException as error:
    print(
        "예상된 최종 실패:",
        type(error).__name__,
    )

=== 네트워크 연결 실패 테스트 ===


{"timestamp": "2026-09-04T20:41:54.194575+00:00", "level": "WARNING", "logger": "api_ingestion", "message": "retryable network exception", "step": "extract", "source": "http://127.0.0.1:1/v1/orders", "attempt": 1}
{"timestamp": "2026-09-04T20:41:56.255570+00:00", "level": "WARNING", "logger": "api_ingestion", "message": "retryable network exception", "step": "extract", "source": "http://127.0.0.1:1/v1/orders", "attempt": 2}


예상된 최종 실패: ConnectTimeout


## Extract pages into an immutable Bronze run

The checkpoint answers, “What source update time was successfully committed?”
It is updated **after** every page and manifest are durable. Moving the checkpoint
too early can permanently skip data after a crash.

A `run_id` separates attempts. We do not overwrite an earlier raw page. The
manifest provides control-plane metadata: counts, checksums, request IDs, and the
watermark used by downstream jobs.


In [6]:
# checkpoint 읽기 → API를 페이지별로 호출 → 각 원본 응답을 Bronze에 저장 → manifest 저장 → 마지막에 checkpoint 갱신 → 실행 결과 반환

def read_checkpoint() -> dict:
    # checkpoint가 없으면 이전 실행 기록이 없는 최초 실행
    if not CHECKPOINT_PATH.exists():
        return {"updated_after": None}

    # 마지막으로 성공한 데이터 위치(high-watermark)를 읽음
    return json.loads(
        CHECKPOINT_PATH.read_text(encoding="utf-8")
    )


def extract_orders(page_size: int = 40) -> dict:
    # 실행 시작 시각과 실행을 구분하기 위한 고유 run_id를 만든다.
    started_at = datetime.now(timezone.utc)
    run_id = started_at.strftime("%Y%m%dT%H%M%S_%fZ")

    # 실행별로 별도의 Bronze 폴더를 만든다.
    # 기존 원본 파일을 덮어쓰지 않기 위한 구조이다.
    run_dir = (
        BRONZE_API
        / f"ingestion_date={started_at.date()}"
        / f"run_id={run_id}"
    )
    run_dir.mkdir(parents=True, exist_ok=False)

    # 이전 실행의 high-watermark를 읽는다.
    prior_checkpoint = read_checkpoint()
    updated_after = prior_checkpoint.get("updated_after")

    # API pagination과 실행 결과 집계에 사용할 초기값
    cursor = None
    page_number = 0
    rows_seen = 0
    max_updated_at = updated_after
    pages = []

    # 이번 extraction 실행이 시작되었다는 로그를 남긴다.
    LOGGER.info(
        "extraction started",
        extra={
            "run_id": run_id,
            "step": "extract",
            "source": API_URL,
        },
    )

    # Session을 사용해 여러 API 요청에서 HTTP 연결을 재사용한다.
    with requests.Session() as session:
        while True:
            # 한 번에 가져올 주문 개수를 설정한다.
            params = {"limit": page_size}

            # 첫 페이지가 아니면 다음 페이지 cursor를 전달한다.
            if cursor is not None:
                params["cursor"] = cursor

            # 이전 실행 기록이 있으면 그 이후에 변경된 주문만 요청한다.
            if updated_after:
                params["updated_after"] = updated_after

            # API를 호출한다. 429, 500 또는 네트워크 오류는 내부에서 재시도한다.
            envelope = get_json_with_retry(
                session,
                API_URL,
                params=params,
            )

            # API 응답에서 현재 페이지의 주문 목록을 꺼낸다.
            records = envelope.get("data")

            # API 계약과 다르게 data가 list가 아니면 즉시 실패시킨다.
            if not isinstance(records, list):
                raise TypeError(
                    "API contract violation: 'data' must be a list"
                )

            # 반환된 주문이 없으면 pagination을 종료한다.
            if not records:
                break

            page_number += 1
            page_path = run_dir / f"page_{page_number:04d}.json"

            # API 응답 전체를 변경하지 않고 Bronze에 안전하게 저장한다.
            atomic_write_json(page_path, envelope)

            # 이번 실행에서 수집한 전체 주문 수를 누적한다.
            rows_seen += len(records)

            # 현재 페이지에서 가장 최신 updated_at을 찾는다.
            page_max = max(
                record["updated_at"]
                for record in records
            )

            # 전체 페이지 중 가장 최신 updated_at을 high-watermark 후보로 유지한다.
            max_updated_at = max(
                filter(None, [max_updated_at, page_max])
            )

            # 파일 경로, 행 수, 크기, checksum 등 감사 정보를 기록한다.
            pages.append(
                {
                    "path": str(
                        page_path.relative_to(PROJECT_ROOT)
                    ),
                    "row_count": len(records),
                    "bytes": page_path.stat().st_size,
                    "sha256": file_sha256(page_path),
                    "request_id": envelope.get("request_id"),
                }
            )

            # 다음 페이지의 cursor를 가져온다.
            cursor = envelope.get("next_cursor")

            # next_cursor가 없으면 마지막 페이지이므로 종료한다.
            if cursor is None:
                break

    # 이번 extraction 실행의 전체 결과를 manifest로 정리한다.
    manifest = {
        "run_id": run_id,
        "started_at": started_at.isoformat(),
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "source": API_URL,
        "starting_watermark": updated_after,
        "ending_watermark": max_updated_at,
        "page_count": page_number,
        "record_count": rows_seen,
        "pages": pages,
        "status": "SUCCESS",
    }

    # manifest를 먼저 저장해 실행 결과와 원본 파일의 증거를 남긴다.
    atomic_write_json(
        run_dir / "manifest.json",
        manifest,
    )

    # 새로운 데이터가 있을 때만 checkpoint를 갱신한다.
    # 원본과 manifest 저장이 성공한 뒤에 갱신해야 데이터 유실을 방지할 수 있다.
    if max_updated_at != updated_after:
        atomic_write_json(
            CHECKPOINT_PATH,
            {
                "updated_after": max_updated_at,
                "committed_run_id": run_id,
            },
        )

    # 실행 완료 로그에 최종 수집 건수를 기록한다.
    LOGGER.info(
        "extraction completed",
        extra={
            "run_id": run_id,
            "step": "extract",
            "record_count": rows_seen,
            "source": API_URL,
        },
    )

    # 호출한 쪽에서 실행 결과를 확인할 수 있도록 manifest를 반환한다.
    return manifest


# 첫 번째 API extraction을 실행하고 결과 manifest를 확인한다.
first_run = extract_orders()
first_run

{"timestamp": "2026-09-04T20:41:58.381602+00:00", "level": "INFO", "logger": "api_ingestion", "message": "extraction started", "run_id": "20260904T204158_380057Z", "step": "extract", "source": "http://127.0.0.1:49521/v1/orders"}
{"timestamp": "2026-09-04T20:41:58.389710+00:00", "level": "WARNING", "logger": "api_ingestion", "message": "retryable HTTP response", "step": "extract", "source": "http://127.0.0.1:49521/v1/orders?limit=40&cursor=40", "attempt": 1}
{"timestamp": "2026-09-04T20:41:58.474262+00:00", "level": "WARNING", "logger": "api_ingestion", "message": "retryable HTTP response", "step": "extract", "source": "http://127.0.0.1:49521/v1/orders?limit=40&cursor=120", "attempt": 1}
{"timestamp": "2026-09-04T20:41:58.652683+00:00", "level": "INFO", "logger": "api_ingestion", "message": "extraction completed", "run_id": "20260904T204158_380057Z", "step": "extract", "record_count": 241, "source": "http://127.0.0.1:49521/v1/orders"}


{'run_id': '20260904T204158_380057Z',
 'started_at': '2026-09-04T20:41:58.380057+00:00',
 'completed_at': '2026-09-04T20:41:58.649324+00:00',
 'source': 'http://127.0.0.1:49521/v1/orders',
 'starting_watermark': None,
 'ending_watermark': '2026-01-03T19:48:00+00:00',
 'page_count': 7,
 'record_count': 241,
 'pages': [{'path': 'lab_data\\bronze\\api\\orders\\ingestion_date=2026-09-04\\run_id=20260904T204158_380057Z\\page_0001.json',
   'row_count': 40,
   'bytes': 27194,
   'sha256': '4f59d7f6515a4952fc22b51f534301a20e028078a6cb78d0f8f1036a4780bd20',
   'request_id': '6dc114fe-1d44-44f7-826d-c2c9c7e1512d'},
  {'path': 'lab_data\\bronze\\api\\orders\\ingestion_date=2026-09-04\\run_id=20260904T204158_380057Z\\page_0002.json',
   'row_count': 40,
   'bytes': 26549,
   'sha256': 'b7ed977a1feeb40f82d0cf659a3697bc4068a65d7638c0961b437c4f89b03aa6',
   'request_id': '59ba1d19-e18b-4123-a973-e2fedeb2a147'},
  {'path': 'lab_data\\bronze\\api\\orders\\ingestion_date=2026-09-04\\run_id=20260904T204

In [7]:
# 모든 원본 데이터를 저장했는가?
# 여러 페이지를 정상적으로 수집했는가?
# checksum을 만들었는가?
# checkpoint를 저장했는가?
# 두 번째 실행에서 같은 데이터를 중복 수집하지 않는가?

# Checkpoint: the first run saved every source record and multiple raw pages.
assert first_run["record_count"] == len(SOURCE_ORDERS)
assert first_run["page_count"] > 1
assert all(len(page["sha256"]) == 64 for page in first_run["pages"])
assert CHECKPOINT_PATH.exists()

# A second run uses the committed high watermark and finds no new records.
second_run = extract_orders()
assert second_run["record_count"] == 0
assert second_run["starting_watermark"] == first_run["ending_watermark"]
print("PASS: raw pages are auditable and the second run is incremental.")

{"timestamp": "2026-09-04T20:41:58.661910+00:00", "level": "INFO", "logger": "api_ingestion", "message": "extraction started", "run_id": "20260904T204158_660797Z", "step": "extract", "source": "http://127.0.0.1:49521/v1/orders"}
{"timestamp": "2026-09-04T20:41:58.666287+00:00", "level": "INFO", "logger": "api_ingestion", "message": "extraction completed", "run_id": "20260904T204158_660797Z", "step": "extract", "record_count": 0, "source": "http://127.0.0.1:49521/v1/orders"}


PASS: raw pages are auditable and the second run is incremental.


## Ingest application logs as a file source

APIs are only one source type. Companies also receive JSONL log files from
services, agents, or object storage delivery jobs. The generator below includes
nested context, a free-text message, an optional stack trace, and one malformed
JSON line. Notebook 02 must parse good records without losing evidence of the bad
record.


In [8]:
# 실행할 때마다 같은 합성 로그가 만들어지도록 random seed를 고정한다.
rng = random.Random(29)

# 로그를 날짜별 Bronze 경로에 GZIP 압축 JSONL 파일로 저장한다.
log_path = (
    BRONZE_LOGS
    / "event_date=2026-01-03"
    / "application_logs.jsonl.gz"
)

# 로그 파일을 저장할 디렉터리가 없으면 생성한다.
log_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# JSONL 파일을 GZIP으로 압축하여 생성한다.
# JSONL은 한 줄에 하나의 JSON event를 저장하는 형식
with gzip.open(
    log_path,
    "wt",
    encoding="utf-8",
) as destination:

    # 총 250개의 합성 애플리케이션 로그를 생성한다.
    for index in range(250):

        # 31번째마다 ERROR를 만들고, 나머지는 INFO 또는 WARN으로 만든다.
        level = (
            "ERROR"
            if index % 31 == 0
            else rng.choice(["INFO", "INFO", "WARN"])
        )

        # 실제 웹 서비스가 생성할 법한 구조화된 JSON 로그를 만든다.
        log_event = {
            "event_id": f"log-{index:05d}",

            # 각 로그가 13초 간격으로 발생하도록 timestamp를 만든다.
            "timestamp": (
                datetime(
                    2026,
                    1,
                    3,
                    tzinfo=timezone.utc,
                )
                + timedelta(seconds=index * 13)
            ).isoformat(),

            # 로그를 생성한 애플리케이션 서비스
            "service": rng.choice(
                ["checkout", "catalog", "payments"]
            ),

            # INFO, WARN 또는 ERROR
            "level": level,

            # 여러 서비스의 로그를 연결해서 추적하기 위한 ID
            "trace_id": (
                f"trace-{rng.randint(1, 90):05d}"
            ),

            # ERROR이면 결제 실패 메시지, 아니면 정상 처리 메시지를 넣는다.
            "message": (
                "Payment authorization failed"
                if level == "ERROR"
                else "Request completed"
            ),

            # 요청과 관련된 상세 정보를 중첩 JSON으로 저장한다.
            "context": {
                "customer_id": (
                    f"cust_{rng.randint(1, 60):04d}"
                ),
                "latency_ms": rng.randint(5, 900),
                "http": {
                    "status": 503 if level == "ERROR" else 200
                },
            },

            # ERROR 로그에만 exception과 stack trace를 추가한다.
            "exception": {
                "type": "PaymentGatewayError",
                "stack_trace": (
                    "PaymentGatewayError: upstream unavailable\n"
                    "  at authorize(payment.py:81)"
                ),
            } if level == "ERROR" else None,
        }

        # 하나의 JSON event를 한 줄로 저장한다.
        destination.write(
            json.dumps(log_event) + "\n"
        )

    # 데이터 품질 처리를 테스트하기 위해 잘못된 JSON 한 줄을 일부러 추가한다.
    destination.write(
        '{"event_id": "broken", "timestamp": '
    )


# 생성한 로그 파일의 경로, 압축 크기, checksum을 기록한다.
log_metadata = {
    "path": str(
        log_path.relative_to(PROJECT_ROOT)
    ),
    "compressed_bytes": log_path.stat().st_size,
    "sha256": file_sha256(log_path),
}

# 생성된 로그 파일의 metadata를 확인한다.
log_metadata

{'path': 'lab_data\\bronze\\logs\\event_date=2026-01-03\\application_logs.jsonl.gz',
 'compressed_bytes': 4280,
 'sha256': '6301c76aae01435d5602236827c8e53c5d93dcd91c6b0d5ece7aa5a7aacc35e7'}

## Your turn

1. Modify the API to return `401` once. Confirm that the client does **not** retry it.
2. Add a page-level record-count reconciliation to the manifest.
3. Simulate a crash immediately before checkpoint commit. Explain what the next run
   will repeat and why downstream deduplication is still necessary.
4. Replace timestamp-only state with `(updated_at, order_id)` state.
5. Write a unit test using a fake session so retry behavior does not need a server.

### Two-minute interview answer

Explain the source contract, transient failures, raw layout, checkpoint commit
point, and evidence that the second run did not duplicate source data. Avoid saying
“S3” unless you actually run the optional S3 extension; this core lab uses a local
object-storage layout.


In [9]:
# Release the local port. The raw files remain available to later notebooks.
api_server.shutdown()
api_server.server_close()
api_thread.join(timeout=2)
print("Local API stopped cleanly.")

Local API stopped cleanly.
